In [7]:
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
import numpy as np
from typing import Dict, List, Tuple
import copy
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

class P60USchedPruner:
    """
    P60-U-sched (Pruning Only) implementation for EfficientDet-Lite0
    P60: 60% pruning ratio
    U-sched: Uniform scheduling
    """

    def __init__(self, model, target_sparsity=0.6, num_iterations=10):
        self.model = model
        self.target_sparsity = target_sparsity
        self.num_iterations = num_iterations
        self.current_sparsity = 0.0
        self.pruning_schedule = self._create_uniform_schedule()

    def _create_uniform_schedule(self) -> List[float]:
        """Create uniform pruning schedule"""
        schedule = []
        step_size = self.target_sparsity / self.num_iterations

        for i in range(self.num_iterations):
            schedule.append(step_size * (i + 1))

        return schedule

    def _get_prunable_layers(self) -> List[Tuple[nn.Module, str]]:
        """Get all prunable layers from the model"""
        prunable_layers = []

        for name, module in self.model.named_modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                # Skip batch normalization related layers
                if 'bn' not in name.lower() and 'norm' not in name.lower():
                    prunable_layers.append((module, 'weight'))

        return prunable_layers

    def _calculate_importance_scores(self, layer: nn.Module) -> torch.Tensor:
        """Calculate importance scores for weights using L1 norm"""
        with torch.no_grad():
            weights = layer.weight.data
            # Use L1 norm as importance metric
            importance = torch.abs(weights)

        return importance

    def _structured_pruning(self, layer: nn.Module, param_name: str, sparsity_ratio: float):
        """Apply structured pruning to a layer"""
        if not hasattr(layer, param_name):
            return

        weight = getattr(layer, param_name)
        if weight is None:
            return

        # Calculate importance scores
        importance = self._calculate_importance_scores(layer)

        # For Conv2d layers, prune entire filters (structured pruning)
        if isinstance(layer, nn.Conv2d):
            # Calculate filter importance (sum across spatial dimensions)
            filter_importance = importance.sum(dim=(1, 2, 3))  # Sum over C_in, H, W
            num_filters_to_prune = int(len(filter_importance) * sparsity_ratio)

            if num_filters_to_prune > 0:
                # Get indices of least important filters
                _, indices = torch.topk(filter_importance,
                                      num_filters_to_prune,
                                      largest=False)

                # Apply structured pruning using PyTorch's prune module
                prune.ln_structured(layer, name=param_name,
                                  amount=num_filters_to_prune,
                                  n=1, dim=0)

        # For Linear layers, use unstructured pruning
        elif isinstance(layer, nn.Linear):
            prune.l1_unstructured(layer, name=param_name, amount=sparsity_ratio)

    def apply_pruning_step(self, iteration: int):
        """Apply pruning for current iteration"""
        if iteration >= len(self.pruning_schedule):
            return

        target_sparsity = self.pruning_schedule[iteration]
        step_sparsity = target_sparsity - self.current_sparsity

        if step_sparsity <= 0:
            return

        print(f"Pruning iteration {iteration + 1}/{self.num_iterations}")
        print(f"Target sparsity: {target_sparsity:.2%}")

        prunable_layers = self._get_prunable_layers()

        for layer, param_name in prunable_layers:
            self._structured_pruning(layer, param_name, step_sparsity)

        self.current_sparsity = target_sparsity

    def remove_pruning_masks(self):
        """Remove pruning masks to make pruning permanent"""
        for name, module in self.model.named_modules():
            if hasattr(module, 'weight_mask'):
                prune.remove(module, 'weight')

    def get_model_sparsity(self) -> float:
        """Calculate current model sparsity"""
        total_params = 0
        zero_params = 0

        for name, module in self.model.named_modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                if hasattr(module, 'weight'):
                    weight = module.weight
                    total_params += weight.numel()
                    zero_params += (weight == 0).sum().item()

        return zero_params / total_params if total_params > 0 else 0.0

    def get_model_size_reduction(self) -> Dict[str, float]:
        """Calculate model size reduction"""
        sparsity = self.get_model_sparsity()

        return {
            'sparsity': sparsity,
            'size_reduction': sparsity,
            'parameters_remaining': 1 - sparsity
        }


In [13]:
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
import numpy as np
from typing import Dict, List, Tuple
import copy
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

class P60USchedPruner:
    """
    P60-U-sched (Pruning Only) implementation for EfficientDet-Lite0
    P60: 60% pruning ratio
    U-sched: Uniform scheduling
    """

    def __init__(self, model, target_sparsity=0.6, num_iterations=10):
        self.model = model
        self.target_sparsity = target_sparsity
        self.num_iterations = num_iterations
        self.current_sparsity = 0.0
        self.pruning_schedule = self._create_uniform_schedule()

    def _create_uniform_schedule(self) -> List[float]:
        """Create uniform pruning schedule"""
        schedule = []
        step_size = self.target_sparsity / self.num_iterations

        for i in range(self.num_iterations):
            schedule.append(step_size * (i + 1))

        return schedule

    def _get_prunable_layers(self) -> List[Tuple[nn.Module, str]]:
        """Get all prunable layers from the model"""
        prunable_layers = []

        for name, module in self.model.named_modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                # Skip batch normalization related layers
                if 'bn' not in name.lower() and 'norm' not in name.lower():
                    prunable_layers.append((module, 'weight'))

        return prunable_layers

    def _calculate_importance_scores(self, layer: nn.Module) -> torch.Tensor:
        """Calculate importance scores for weights using L1 norm"""
        with torch.no_grad():
            weights = layer.weight.data
            # Use L1 norm as importance metric
            importance = torch.abs(weights)

        return importance

    def _structured_pruning(self, layer: nn.Module, param_name: str, sparsity_ratio: float):
        """Apply structured pruning to a layer"""
        if not hasattr(layer, param_name):
            return

        weight = getattr(layer, param_name)
        if weight is None:
            return

        # Calculate importance scores
        importance = self._calculate_importance_scores(layer)

        # For Conv2d layers, prune entire filters (structured pruning)
        if isinstance(layer, nn.Conv2d):
            # Calculate filter importance (sum across spatial dimensions)
            filter_importance = importance.sum(dim=(1, 2, 3))  # Sum over C_in, H, W
            num_filters_to_prune = int(len(filter_importance) * sparsity_ratio)

            if num_filters_to_prune > 0:
                # Get indices of least important filters
                _, indices = torch.topk(filter_importance,
                                      num_filters_to_prune,
                                      largest=False)

                # Apply structured pruning using PyTorch's prune module
                prune.ln_structured(layer, name=param_name,
                                  amount=num_filters_to_prune,
                                  n=1, dim=0)

        # For Linear layers, use unstructured pruning
        elif isinstance(layer, nn.Linear):
            prune.l1_unstructured(layer, name=param_name, amount=sparsity_ratio)

    def apply_pruning_step(self, iteration: int):
        """Apply pruning for current iteration"""
        if iteration >= len(self.pruning_schedule):
            return

        target_sparsity = self.pruning_schedule[iteration]
        step_sparsity = target_sparsity - self.current_sparsity

        if step_sparsity <= 0:
            return

        print(f"Pruning iteration {iteration + 1}/{self.num_iterations}")
        print(f"Target sparsity: {target_sparsity:.2%}")

        prunable_layers = self._get_prunable_layers()

        for layer, param_name in prunable_layers:
            self._structured_pruning(layer, param_name, step_sparsity)

        self.current_sparsity = target_sparsity

    def remove_pruning_masks(self):
        """Remove pruning masks to make pruning permanent"""
        for name, module in self.model.named_modules():
            if hasattr(module, 'weight_mask'):
                prune.remove(module, 'weight')

    def get_model_sparsity(self) -> float:
        """Calculate current model sparsity"""
        total_params = 0
        zero_params = 0

        for name, module in self.model.named_modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                if hasattr(module, 'weight'):
                    weight = module.weight
                    total_params += weight.numel()
                    zero_params += (weight == 0).sum().item()

        return zero_params / total_params if total_params > 0 else 0.0

    def get_model_size_reduction(self) -> Dict[str, float]:
        """Calculate model size reduction"""
        sparsity = self.get_model_sparsity()

        return {
            'sparsity': sparsity,
            'size_reduction': sparsity,
            'parameters_remaining': 1 - sparsity
        }

class EfficientDetLite0Trainer:
    """Training utilities for pruned model"""

    def __init__(self, model, device='cuda'):
        self.model = model
        self.device = device
        self.model.to(device)

    def fine_tune(self, train_loader: DataLoader, val_loader: DataLoader,
                  num_epochs: int = 5, lr: float = 1e-4):
        """Fine-tune the pruned model"""
        optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)
        # Note: This is a placeholder loss calculation for a dummy model and dataset.
        # For a real object detection task with EfficientDet-Lite0,
        # you would use a loss function like RetinaNetLoss or similar,
        # which compares predicted bounding boxes and class logits to ground truth.
        # The dummy dataset provides integer labels, while the dummy model outputs
        # a 1000-dimensional vector. MSELoss is not appropriate here due to shape mismatch.
        # We use a dummy loss calculation that avoids the RuntimeError.
        # Replace with appropriate loss for object detection when using a real model/data.
        # criterion = nn.MSELoss() # Original problematic loss

        self.model.train()

        for epoch in range(num_epochs):
            total_loss = 0.0
            num_batches = 0

            for batch_idx, (data, targets) in enumerate(train_loader):
                data = data.to(self.device)
                # targets are not used in this dummy loss calculation to avoid shape issues
                # targets = targets.to(self.device) # Original line

                optimizer.zero_grad()
                outputs = self.model(data)

                # Dummy loss calculation to avoid RuntimeError
                # In a real scenario, 'outputs' would contain bounding box predictions and class logits
                # and 'targets' would contain ground truth boxes and labels.
                # A proper object detection loss would be calculated here.
                # Example placeholder: sum of outputs (meaningless for real task)
                loss = torch.sum(outputs) * 0.001 # Scale down to prevent large values

                loss.backward()
                optimizer.step()

                total_loss += loss.item()
                num_batches += 1

                if batch_idx % 100 == 0:
                    print(f'Epoch {epoch+1}/{num_epochs}, '
                          f'Batch {batch_idx}/{len(train_loader)}, '
                          f'Dummy Loss: {loss.item():.6f}')

            avg_loss = total_loss / num_batches
            print(f'Epoch {epoch+1} completed. Average Dummy Loss: {avg_loss:.6f}')

            # Validation (using dummy loss)
            if val_loader is not None:
                val_loss = self.validate(val_loader) # Pass validation data
                print(f'Validation Dummy Loss: {val_loss:.6f}')

    def validate(self, val_loader: DataLoader):
        """Validate the model"""
        self.model.eval()
        total_loss = 0.0
        num_batches = 0

        with torch.no_grad():
            for data, targets in val_loader:
                data = data.to(self.device)
                # targets are not used in this dummy loss calculation
                # targets = targets.to(self.device) # Original line
                outputs = self.model(data)
                # Dummy loss calculation
                loss = torch.sum(outputs) * 0.001 # Scale down

                total_loss += loss.item()
                num_batches += 1

        self.model.train()
        return total_loss / num_batches

def main_pruning_pipeline(model, train_loader=None, val_loader=None):
    """Main pruning pipeline for EfficientDet-Lite0"""

    print("Starting P60-U-sched Pruning for EfficientDet-Lite0")
    print("="*50)

    # Initialize pruner
    # Use a lower target sparsity for faster demonstration if needed, e.g., 0.2
    pruner = P60USchedPruner(model, target_sparsity=0.6, num_iterations=10)
    trainer = EfficientDetLite0Trainer(model)

    # Get initial model stats
    initial_sparsity = pruner.get_model_sparsity()
    print(f"Initial model sparsity: {initial_sparsity:.2%}")

    # Apply gradual pruning
    for iteration in range(pruner.num_iterations):
        print(f"\nPruning Step {iteration + 1}")
        print("-" * 30)

        # Apply pruning step
        pruner.apply_pruning_step(iteration)

        # Fine-tune after pruning (if data loaders are provided)
        if train_loader is not None:
            print("Fine-tuning after pruning...")
            # Use fewer epochs for faster demonstration if needed, e.g., num_epochs=1
            trainer.fine_tune(train_loader, val_loader, num_epochs=2, lr=1e-5)

        # Print current statistics
        current_sparsity = pruner.get_model_sparsity()
        print(f"Current sparsity: {current_sparsity:.2%}")

    # Remove pruning masks to make pruning permanent
    pruner.remove_pruning_masks()

    # Final statistics
    final_stats = pruner.get_model_size_reduction()
    print("\nFinal Pruning Results:")
    print("="*50)
    print(f"Final sparsity: {final_stats['sparsity']:.2%}")
    print(f"Model size reduction: {final_stats['size_reduction']:.2%}")
    print(f"Parameters remaining: {final_stats['parameters_remaining']:.2%}")

    return model, final_stats

In [14]:
# Example usage with comprehensive evaluation
if __name__ == "__main__":
    # Note: Replace this with actual EfficientDet-Lite0 model loading
    # from torchvision or timm library

    # Dummy model for demonstration
    class DummyEfficientDetLite0(nn.Module):
        def __init__(self, num_classes=1000):
            super().__init__()
            self.backbone = nn.Sequential(
                nn.Conv2d(3, 32, 3, padding=1),
                nn.BatchNorm2d(32),
                nn.ReLU(),
                nn.Conv2d(32, 64, 3, padding=1),
                nn.BatchNorm2d(64),
                nn.ReLU(),
                nn.Conv2d(64, 128, 3, padding=1),
                nn.BatchNorm2d(128),
                nn.ReLU(),
                nn.AdaptiveAvgPool2d((1, 1))
            )
            self.classifier = nn.Linear(128, num_classes)

        def forward(self, x):
            x = self.backbone(x)
            x = x.view(x.size(0), -1)
            return self.classifier(x)

    # Create dummy dataset for demonstration
    class DummyDataset(torch.utils.data.Dataset):
        def __init__(self, num_samples=1000, num_classes=1000):
            self.num_samples = num_samples
            self.num_classes = num_classes

        def __len__(self):
            return self.num_samples

        def __getitem__(self, idx):
            # Generate random image and label
            image = torch.randn(3, 320, 320)
            label = torch.randint(0, self.num_classes, (1,)).item()
            return image, label

    # Placeholder functions for evaluation metrics (to avoid NameError)
    # In a real scenario, these would be implemented based on your task (e.g., object detection metrics)
    def calculate_f1_metrics(model, dataloader, device):
        """Placeholder for F1 score calculation."""
        print("Calculating dummy F1 metrics...")
        # Dummy implementation: return fixed values
        return {'f1_macro': 0.5, 'accuracy': 0.6}

    def get_model_size_metrics(model):
        """Placeholder for model size calculation."""
        print("Calculating dummy model size metrics...")
        # Dummy implementation: return fixed values
        param_size = sum(p.numel() for p in model.parameters())
        buffer_size = sum(b.numel() for b in model.buffers())
        total_size = param_size + buffer_size
        size_in_bytes = total_size * 4 # Assuming float32
        size_in_mb = size_in_bytes / (1024 * 1024)
        return {'pytorch_model_size_mb': size_in_mb, 'total_parameters': param_size}

    def get_fps_metrics(model, dataloader, device, num_samples=100):
        """Placeholder for FPS calculation."""
        print("Calculating dummy FPS metrics...")
        # Dummy implementation: return fixed values
        import time
        model.eval()
        start_time = time.time()
        with torch.no_grad():
            for i, (data, targets) in enumerate(dataloader):
                if i * dataloader.batch_size >= num_samples:
                    break
                data = data.to(device)
                _ = model(data)
        end_time = time.time()
        total_time = end_time - start_time
        avg_inference_time_ms = (total_time / num_samples) * 1000
        fps = num_samples / total_time if total_time > 0 else 0
        return {'fps': fps, 'avg_inference_time_ms': avg_inference_time_ms}

    def convert_and_evaluate_tflite(model, test_loader, device, tflite_path):
        """Placeholder for TFLite conversion and evaluation."""
        print(f"Converting and evaluating dummy TFLite model at {tflite_path}...")
        # Dummy implementation: return fixed values and save a dummy file
        dummy_tflite_size_mb = get_model_size_metrics(model)['pytorch_model_size_mb'] * 0.5 # Assume 50% reduction
        dummy_tflite_fps = get_fps_metrics(model, test_loader, device)['fps'] * 1.5 # Assume 50% improvement

        # Save a dummy file
        with open(tflite_path, 'wb') as f:
            f.write(b'dummy tflite content')

        return {
            'model_size_mb': dummy_tflite_size_mb,
            'fps': dummy_tflite_fps,
            'inference_time_ms': 1000 / dummy_tflite_fps if dummy_tflite_fps > 0 else float('inf')
            }

    # Create datasets and dataloaders
    print("Creating dummy datasets for evaluation...")
    train_dataset = DummyDataset(num_samples=5000, num_classes=1000)
    test_dataset = DummyDataset(num_samples=1000, num_classes=1000)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
    val_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)  # Using test as val for demo

    # Create model
    original_model = DummyEfficientDetLite0(num_classes=1000)
    pruned_model_input = copy.deepcopy(original_model) # Model to be pruned

    print("Starting EfficientDet-Lite0 Pruning with Comprehensive Performance Evaluation")
    print("="*80)

    # --- Perform comprehensive evaluation ---
    results = {
        'original_results': {},
        'pruned_results': {},
        'tflite_results': None,
        'comparison': {}
    }

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    original_model.to(device)
    pruned_model_input.to(device)


    # 1. Evaluate Original Model
    print("\nEvaluating Original Model Performance...")
    results['original_results']['size_metrics'] = get_model_size_metrics(original_model)
    results['original_results']['fps_metrics'] = get_fps_metrics(original_model, test_loader, device)
    results['original_results']['f1_metrics'] = calculate_f1_metrics(original_model, test_loader, device)


    # 2. Apply Pruning Pipeline
    print("\nApplying Pruning Pipeline...")
    # Call the existing main_pruning_pipeline function
    pruned_model, final_pruning_stats = main_pruning_pipeline(
        model=pruned_model_input, # Pass the model to be pruned
        train_loader=train_loader,
        val_loader=val_loader
    )

    # Move pruned model to device after pruning
    pruned_model.to(device)

    # 3. Evaluate Pruned Model
    print("\nEvaluating Pruned Model Performance...")
    results['pruned_results']['size_metrics'] = get_model_size_metrics(pruned_model)
    results['pruned_results']['fps_metrics'] = get_fps_metrics(pruned_model, test_loader, device)
    results['pruned_results']['f1_metrics'] = calculate_f1_metrics(pruned_model, test_loader, device)
    # Add training time metrics (dummy values or capture from fine-tuning if implemented)
    results['pruned_results']['training_metrics'] = {
        'training_time_per_epoch': 100.0, # Dummy value
        'training_time_per_batch': 1.0, # Dummy value
        'samples_per_second': 32.0, # Dummy value
        'cpu_efficiency': 0.8, # Dummy value
        'avg_gpu_utilization': 70.0 if device == 'cuda' else None # Dummy value
    }


    # 4. Convert and Evaluate TFLite Model
    print("\nConverting and Evaluating TFLite Model...")
    tflite_path = 'efficientdet_lite0_pruned_p60_quantized.tflite'
    results['tflite_results'] = convert_and_evaluate_tflite(pruned_model, test_loader, device, tflite_path)


    # 5. Calculate Comparison Metrics
    print("\nCalculating Comparison Metrics...")
    original_size = results['original_results']['size_metrics']['pytorch_model_size_mb']
    pruned_size = results['pruned_results']['size_metrics']['pytorch_model_size_mb']
    original_fps = results['original_results']['fps_metrics']['fps']
    pruned_fps = results['pruned_results']['fps_metrics']['fps']
    original_params = results['original_results']['size_metrics']['total_parameters']
    pruned_params = results['pruned_results']['size_metrics']['total_parameters']
    original_f1 = results['original_results']['f1_metrics']['f1_macro']
    pruned_f1 = results['pruned_results']['f1_metrics']['f1_macro']

    results['comparison']['size_reduction_percent'] = (1 - pruned_size / original_size) * 100 if original_size > 0 else 0
    results['comparison']['fps_improvement_percent'] = ((pruned_fps / original_fps) - 1) * 100 if original_fps > 0 else 0
    results['comparison']['parameter_reduction_percent'] = (1 - pruned_params / original_params) * 100 if original_params > 0 else 0
    results['comparison']['f1_score_change_percent'] = ((pruned_f1 / original_f1) - 1) * 100 if original_f1 > 0 else 0
    # Add training speedup if applicable and calculable
    results['comparison']['training_speedup_percent'] = 0 # Dummy value


    # Additional detailed analysis
    print("\n" + "="*80)
    print("DETAILED PERFORMANCE METRICS BREAKDOWN")
    print("="*80)

    if results['original_results'] and results['pruned_results']:
        original = results['original_results']
        pruned = results['pruned_results']

        print("\n📊 F1-SCORE METRICS:")
        print("-"*40)
        if 'f1_metrics' in original and 'f1_metrics' in pruned:
            print(f"Original F1-Score (Macro): {original['f1_metrics']['f1_macro']:.4f}")
            print(f"Pruned F1-Score (Macro):   {pruned['f1_metrics']['f1_macro']:.4f}")
            # Ensure division by zero is handled
            f1_change = ((pruned['f1_metrics']['f1_macro']/original['f1_metrics']['f1_macro'])-1)*100 if original['f1_metrics']['f1_macro'] != 0 else float('inf')
            print(f"F1-Score Change: {f1_change:.2f}%")
            print(f"Original Accuracy: {original['f1_metrics']['accuracy']:.4f}")
            print(f"Pruned Accuracy:   {pruned['f1_metrics']['accuracy']:.4f}")

        print("\n⏱️ TRAINING TIME METRICS:")
        print("-"*40)
        if 'training_metrics' in pruned:
            training = pruned['training_metrics']
            print(f"Training Time per Epoch: {training.get('training_time_per_epoch', 0):.2f} seconds")
            print(f"Training Time per Batch: {training.get('training_time_per_batch', 0):.4f} seconds")
            print(f"Training Samples/Second: {training.get('samples_per_second', 0):.1f}")
            print(f"CPU Efficiency: {training.get('cpu_efficiency', 0)*100:.1f}%")
            if 'avg_gpu_utilization' in training and training['avg_gpu_utilization'] is not None:
                print(f"Average GPU Utilization: {training['avg_gpu_utilization']:.1f}%")

        print("\n💾 MODEL SIZE METRICS:")
        print("-"*40)
        print(f"Original Model Size: {original['size_metrics']['pytorch_model_size_mb']:.2f} MB")
        print(f"Pruned Model Size:   {pruned['size_metrics']['pytorch_model_size_mb']:.2f} MB")
        # Ensure division by zero is handled
        size_reduction = (1-pruned['size_metrics']['pytorch_model_size_mb']/original['size_metrics']['pytorch_model_size_mb'])*100 if original['size_metrics']['pytorch_model_size_mb'] != 0 else float('inf')
        print(f"Size Reduction:      {size_reduction:.1f}%")
        print(f"Original Parameters: {original['size_metrics']['total_parameters']:,}")
        print(f"Pruned Parameters:   {pruned['size_metrics']['total_parameters']:,}")
        # Ensure division by zero is handled
        param_reduction = (1-pruned['size_metrics']['total_parameters']/original['size_metrics']['total_parameters'])*100 if original['size_metrics']['total_parameters'] != 0 else float('inf')
        print(f"Parameter Reduction: {param_reduction:.1f}%")

        if results['tflite_results']:
            tflite = results['tflite_results']
            print(f"TFLite Model Size:   {tflite['model_size_mb']:.2f} MB")
            # Ensure division by zero is handled
            tflite_total_reduction = (1-tflite['model_size_mb']/original['size_metrics']['pytorch_model_size_mb'])*100 if original['size_metrics']['pytorch_model_size_mb'] != 0 else float('inf')
            print(f"Total Size Reduction: {tflite_total_reduction:.1f}%")

        print("\n🚀 FPS PERFORMANCE METRICS:")
        print("-"*40)
        print(f"Original FPS:        {original['fps_metrics']['fps']:.1f}")
        print(f"Pruned FPS:          {pruned['fps_metrics']['fps']:.1f}")
        # Ensure division by zero is handled
        fps_improvement = ((pruned['fps_metrics']['fps']/original['fps_metrics']['fps'])-1)*100 if original['fps_metrics']['fps'] != 0 else float('inf')
        print(f"FPS Improvement:     {fps_improvement:.1f}%")
        print(f"Original Inference:  {original['fps_metrics']['avg_inference_time_ms']:.2f} ms")
        print(f"Pruned Inference:    {pruned['fps_metrics']['avg_inference_time_ms']:.2f} ms")
        # Ensure division by zero is handled
        speed_improvement = (1-pruned['fps_metrics']['avg_inference_time_ms']/original['fps_metrics']['avg_inference_time_ms'])*100 if original['fps_metrics']['avg_inference_time_ms'] != 0 else float('inf')
        print(f"Speed Improvement:   {speed_improvement:.1f}%")

        if results['tflite_results']:
            tflite = results['tflite_results']
            print(f"TFLite FPS:          {tflite['fps']:.1f}")
            print(f"TFLite Inference:    {tflite['inference_time_ms']:.2f} ms")
            # Ensure division by zero is handled
            tflite_vs_original_fps_improvement = ((tflite['fps']/original['fps_metrics']['fps'])-1)*100 if original['fps_metrics']['fps'] != 0 else float('inf')
            print(f"TFLite vs Original:  {tflite_vs_original_fps_improvement:.1f}% FPS improvement")

    # Save evaluation results
    if results['original_results'] and results['pruned_results']:
        evaluation_summary = {
            'pruning_method': 'P60-U-sched',
            'target_sparsity': 0.6,
            'original_model': {
                'size_mb': results['original_results']['size_metrics']['pytorch_model_size_mb'],
                'parameters': results['original_results']['size_metrics']['total_parameters'],
                'fps': results['original_results']['fps_metrics']['fps'],
                'inference_time_ms': results['original_results']['fps_metrics']['avg_inference_time_ms'],
            },
            'pruned_model': {
                'size_mb': results['pruned_results']['size_metrics']['pytorch_model_size_mb'],
                'parameters': results['pruned_results']['size_metrics']['total_parameters'],
                'fps': results['pruned_results']['fps_metrics']['fps'],
                'inference_time_ms': results['pruned_results']['fps_metrics']['avg_inference_time_ms'],
                'training_time_per_epoch': results['pruned_results']['training_metrics']['training_time_per_epoch'] if 'training_metrics' in results['pruned_results'] else None
            },
            'improvements': {
                'size_reduction_percent': results['comparison'].get('size_reduction_percent', 0),
                'fps_improvement_percent': results['comparison'].get('fps_improvement_percent', 0),
                'parameter_reduction_percent': results['comparison'].get('parameter_reduction_percent', 0)
            }
        }

        if 'f1_metrics' in results['original_results'] and 'f1_metrics' in results['pruned_results']:
            evaluation_summary['original_model']['f1_score'] = results['original_results']['f1_metrics']['f1_macro']
            evaluation_summary['pruned_model']['f1_score'] = results['pruned_results']['f1_metrics']['f1_macro']
            evaluation_summary['improvements']['f1_score_change_percent'] = results['comparison'].get('f1_score_change_percent', 0)

        if results['tflite_results']:
            evaluation_summary['tflite_model'] = results['tflite_results']
            evaluation_summary['improvements']['tflite_size_reduction_percent'] = (1-results['tflite_results']['model_size_mb']/results['original_results']['size_metrics']['pytorch_model_size_mb'])*100 if results['original_results']['size_metrics']['pytorch_model_size_mb'] != 0 else float('inf')


        # Save to JSON file
        import json
        with open('model_evaluation_results.json', 'w') as f:
            json.dump(evaluation_summary, f, indent=2)

        print(f"\n💾 Evaluation results saved to 'model_evaluation_results.json'")

    print("\n✅ Comprehensive evaluation completed successfully!")
    print("\nFiles generated:")
    # Update generated file list based on the implemented logic
    print("- model_evaluation_results.json (Detailed evaluation metrics)")
    print("- efficientdet_lite0_pruned_p60_quantized.tflite (Quantized TFLite model - dummy saved)")


    print("\n📋 Quick Summary:")
    if results['comparison']:
        comp = results['comparison']
        print(f"🔹 Model size reduced by {comp.get('size_reduction_percent', 0):.1f}%")
        print(f"🔹 Inference speed improved by {comp.get('fps_improvement_percent', 0):.1f}%")
        if 'f1_score_change_percent' in comp:
            change_symbol = "📈" if comp['f1_score_change_percent'] >= 0 else "📉"
            print(f"🔹 F1-Score changed by {change_symbol} {abs(comp['f1_score_change_percent']):.1f}%")
        if 'training_speedup_percent' in comp:
            print(f"🔹 Training speed improved by {comp['training_speedup_percent']:.1f}%")

Creating dummy datasets for evaluation...
Starting EfficientDet-Lite0 Pruning with Comprehensive Performance Evaluation

Evaluating Original Model Performance...
Calculating dummy model size metrics...
Calculating dummy FPS metrics...
Calculating dummy F1 metrics...

Applying Pruning Pipeline...
Starting P60-U-sched Pruning for EfficientDet-Lite0
Initial model sparsity: 0.00%

Pruning Step 1
------------------------------
Pruning iteration 1/10
Target sparsity: 6.00%
Fine-tuning after pruning...
Epoch 1/2, Batch 0/157, Dummy Loss: 0.009650
Epoch 1/2, Batch 100/157, Dummy Loss: -1.479062
Epoch 1 completed. Average Dummy Loss: -1.140799
Validation Dummy Loss: -2.219139
Epoch 2/2, Batch 0/157, Dummy Loss: -2.331518
Epoch 2/2, Batch 100/157, Dummy Loss: -3.834767
Epoch 2 completed. Average Dummy Loss: -3.480294
Validation Dummy Loss: -4.512852
Current sparsity: 5.70%

Pruning Step 2
------------------------------
Pruning iteration 2/10
Target sparsity: 12.00%
Fine-tuning after pruning...
E